## ⚙️ Step 0: Install Dependencies
Run this once per session. The `%%capture` suppresses noisy install output.

In [1]:
%%capture
# Install Unsloth with Colab-optimized extras
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir --no-deps unsloth_zoo
!pip install datasets wandb trl peft transformers bitsandbytes python-dotenv

## 🔐 Step 1: Authentication

We use **Colab Secrets** (the 🔑 key icon in the left sidebar) to securely store API tokens — no `.env` file needed.

**Before running this cell:**
1. Click the 🔑 **Secrets** icon in the left sidebar
2. Add a secret named `HUGGINGFACE_TOKEN` with your HF token (starts with `hf_`)
3. Add a secret named `WANDB_API_KEY` with your W&B API key (optional, for run tracking)

In [2]:
from huggingface_hub import login
import wandb
import os

# --- Load secrets from Colab Secrets (recommended) ---
try:
    from google.colab import userdata
    hf_token  = userdata.get('HUGGINGFACE_TOKEN')
    wb_token  = userdata.get('WANDB_API_KEY')
    print("✅ Loaded secrets from Colab Secrets.")
except Exception:
    # Fallback: read from environment variables if not in Colab
    hf_token = os.environ.get('HUGGINGFACE_TOKEN')
    wb_token = os.environ.get('WANDB_API_KEY')
    print("ℹ️  Loaded from environment variables (non-Colab environment).")

# Hugging Face login
if hf_token and hf_token.startswith('hf_'):
    login(token=hf_token)
    print("✅ Hugging Face: Logged in successfully.")
else:
    print("❌ CRITICAL: HUGGINGFACE_TOKEN not found or invalid.")
    print("   Add it via the 🔑 Secrets panel in the left sidebar.")

# Weights & Biases login (optional)
if wb_token:
    wandb.login(key=wb_token)
    run = wandb.init(
        project='Fine-tune-DeepSeek-R1-Lab-Tests',
        job_type='training',
        anonymous='allow'
    )
    print("✅ W&B: Tracking enabled.")
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print("ℹ️  W&B: No key found — tracking disabled.")

✅ Loaded secrets from Colab Secrets.
✅ Hugging Face: Logged in successfully.


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nguumatn (nguumatn-entrick-information-systems) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


✅ W&B: Tracking enabled.


## 🖥️ Step 2: GPU Check
Verify the GPU Colab assigned. A T4 gives ~15 GB free VRAM — plenty for this model in 4-bit.

In [3]:
import torch

if torch.cuda.is_available():
    device_name   = torch.cuda.get_device_name(0)
    total_memory  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {device_name}")
    print(f"   Total VRAM: {total_memory:.2f} GB")
    print(f"   PyTorch:    {torch.__version__}")
    print(f"   CUDA:       {torch.version.cuda}")
else:
    print("❌ No GPU detected!")
    print("   Go to Runtime > Change runtime type > T4 GPU and re-run.")
    raise RuntimeError("GPU required. Please enable a GPU runtime.")

✅ GPU: Tesla T4
   Total VRAM: 15.64 GB
   PyTorch:    2.10.0+cu128
   CUDA:       12.8


#### **CRITICAL ACTION REQUIRED: Enable GPU and Restart Runtime**

The previous execution failed because no GPU was detected. This notebook requires a GPU to run `unsloth`.

**To resolve this, you MUST perform the following steps:**
1.  Go to `Runtime` > `Change runtime type` in the Colab menu.
2.  Select `T4 GPU` as the hardware accelerator and click `Save`.
3.  After the runtime has restarted, **run all cells from the beginning of the notebook** (from "Step 0: Install Dependencies").

This will ensure `unsloth` can properly detect and utilize the GPU.

## 🤖 Step 3: Load Model & Tokenizer
Using Unsloth's `FastLanguageModel` with **4-bit quantization** to fit the 8B model into Colab's T4 GPU.

In [11]:
import os
from unsloth import FastLanguageModel

MODEL_NAME     = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
max_seq_length = 4096  # Increase to 4096 for longer contexts (uses more VRAM)
dtype          = None  # Auto-detect: bf16 on A100, fp16 on T4
load_in_4bit   = True  # Essential to fit 8B into ~15 GB VRAM

def load_model(use_modelscope=False):
    if use_modelscope:
        os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
        print("🔄 Retrying with ModelScope mirror...")
    else:
        os.environ.pop('UNSLOTH_USE_MODELSCOPE', None)
        print("📡 Attempting to load from HuggingFace...")

    return FastLanguageModel.from_pretrained(
        model_name     = MODEL_NAME,
        max_seq_length = max_seq_length,
        dtype          = dtype,
        load_in_4bit   = load_in_4bit,
        token          = hf_token,
    )

# Try HuggingFace first, fall back to ModelScope on timeout
try:
    model, tokenizer = load_model(use_modelscope=False)
except TimeoutError as e:
    print(f"\n⚠️  HuggingFace timed out: {e}")
    print("   Check status at https://status.huggingface.co")
    model, tokenizer = load_model(use_modelscope=True)

print(f"\n✅ Model loaded: {MODEL_NAME}")
print(f"   Params: {sum(p.numel() for p in model.parameters()):,} total")

📡 Attempting to load from HuggingFace...
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/deepseek-r1-distill-llama-8b-unsloth-bnb-4bit as a legacy tokenizer.



✅ Model loaded: deepseek-ai/DeepSeek-R1-Distill-Llama-8B
   Params: 4,628,680,704 total


## 🔧 Step 4: Configure LoRA Adapters
LoRA only trains a tiny fraction of parameters (~1-5%), so VRAM usage stays low.

In [12]:
model = FastLanguageModel.get_peft_model(
    model,
    r                        = 16,   # LoRA rank — higher = more expressive, more VRAM
    target_modules           = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha               = 16,
    lora_dropout             = 0,    # 0 is recommended by Unsloth
    bias                     = "none",
    use_gradient_checkpointing = "unsloth",  # Saves VRAM for long contexts
    random_state             = 3407,
    use_rslora               = False,
    loftq_config             = None,
)

print(f"✅ LoRA adapters attached.")
print(f"   Trainable params after LoRA: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

✅ LoRA adapters attached.
   Trainable params after LoRA: 41,943,040


## 📂 Step 5: Load & Prepare the Dataset

The dataset `fine_tuning_lab_tests.jsonl` needs to be uploaded to this Colab session.

**Option A (recommended) — upload from your PC:**
```python
from google.colab import files
files.upload()  # select fine_tuning_lab_tests.jsonl
```

**Option B — load directly from a GitHub raw URL** (if the file is in your repo):
```python
!wget https://raw.githubusercontent.com/YOUR_USERNAME/YOUR_REPO/main/fine_tuning_lab_tests.jsonl
```

Run whichever option applies, then run the next cell.

In [13]:
# --- Option A: Upload from local PC ---
from google.colab import files
print("Select your 'fine_tuning_lab_tests.jsonl' file:")
uploaded     = files.upload()
DATASET_FILE = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {DATASET_FILE}")

Select your 'fine_tuning_lab_tests.jsonl' file:


Saving fine_tuning_lab_tests.jsonl to fine_tuning_lab_tests (1).jsonl

✅ Uploaded: fine_tuning_lab_tests (1).jsonl


In [ ]:
# --- Option B: Download from GitHub (replace URL with your own) ---
# DATASET_URL = "https://raw.githubusercontent.com/YOUR_USERNAME/YOUR_REPO/main/fine_tuning_lab_tests.jsonl"
# !wget -q {DATASET_URL}
# DATASET_FILE = "fine_tuning_lab_tests.jsonl"
# print(f"✅ Downloaded: {DATASET_FILE}")

In [14]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=DATASET_FILE, split="train")
print(f"✅ Dataset loaded: {len(dataset)} examples")
print("   Sample keys:", dataset.column_names)

# DeepSeek-R1 specific prompt format (uses <think> tags for chain-of-thought)
prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
{system}

### Question:
{user}

### Response:
<think>
{thought}</think>
{response}"""

def formatting_prompts_func(examples):
    texts = []
    for messages in examples['messages']:
        system    = messages[0]['content']
        user      = messages[1]['content']
        assistant = messages[2]['content']

        thought_process = (
            "Analyzing the lab result against the reference range... "
            "Determining if the value is high, low, or normal... "
            "Formulating clinical recommendations based on standard medical guidelines."
        )

        text = prompt_style.format(
            system   = system,
            user     = user,
            thought  = thought_process,
            response = assistant
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"✅ Dataset formatted. Sample:")
print(dataset[0]['text'][:300], "...")

Generating train split: 0 examples [00:00, ? examples/s]

✅ Dataset loaded: 100 examples
   Sample keys: ['messages']


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

✅ Dataset formatted. Sample:
Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

# ...


## 🏋️ Step 6: Fine-Tuning

Training settings are tuned for Colab free tier (T4, ~15 GB VRAM).

| Setting | Value | Reason |
|---|---|---|
| `per_device_train_batch_size` | 2 | Keeps VRAM under limit |
| `gradient_accumulation_steps` | 4 | Effective batch = 8 |
| `max_steps` | 60 | ~5 min on T4; increase for full training |
| `optim` | `adamw_8bit` | Saves ~4 GB VRAM vs standard Adam |

In [15]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = max_seq_length,
    dataset_num_proc   = 2,
    args = TrainingArguments(
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 4,
        warmup_steps                 = 5,
        max_steps                    = 60,        # ⬆️ Set to -1 and use num_train_epochs=3 for full training
        # num_train_epochs             = 3,        # Uncomment for full training
        learning_rate                = 2e-4,
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = 10,
        optim                        = "adamw_8bit",
        weight_decay                 = 0.01,
        lr_scheduler_type            = "linear",
        seed                         = 3407,
        output_dir                   = "/content/outputs",
        report_to                    = "wandb" if wb_token else "none",
    ),
)

print("🚀 Starting training...")
trainer_stats = trainer.train()
print(f"\n✅ Training complete!")
print(f"   Runtime: {trainer_stats.metrics['train_runtime']:.0f} seconds")
print(f"   Samples/sec: {trainer_stats.metrics['train_samples_per_second']:.2f}")

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/100 [00:00<?, ? examples/s]

🚀 Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 5 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
10,2.103519
20,0.413486
30,0.284152
40,0.157010
50,0.122776
60,0.097672


train/epoch,▁▂▄▅▇██
train/global_step,▁▂▄▅▇██
train/grad_norm,█▃▂▁▁▁
train/learning_rate,█▇▅▄▂▁
train/loss,█▂▂▁▁▁
total_flos,4296280851087360.0
train/epoch,4.64
train/global_step,60
train/grad_norm,0.4203
train/learning_rate,0.0
train/loss,0.09767



✅ Training complete!
   Runtime: 297 seconds
   Samples/sec: 1.61


## 🧪 Step 7: Inference / Testing
Test the fine-tuned model on a sample lab result.

In [16]:
import json
import re
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

# ── 1. Define the test ───────────────────────────────────────────
test_system = "You are a medical laboratory assistant. Analyze lab results, identify abnormalities based on reference ranges, and provide brief explanations for healthcare professionals."
test_user   = "Glucose (Fasting). Result: 155 mg/dL. Ref Range: 70-99 mg/dL."

inference_prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
{test_system}

### Question:
{test_user}

### Response:
<think>"""

# ── 2. Run inference ─────────────────────────────────────────────
inputs  = tokenizer([inference_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(
    input_ids          = inputs.input_ids,
    attention_mask     = inputs.attention_mask,
    max_new_tokens     = 300,
    use_cache          = True,
    do_sample          = True,
    temperature        = 0.7,
    repetition_penalty = 1.3,
    eos_token_id       = tokenizer.eos_token_id,
)
raw_output    = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
response_text = raw_output.split("### Response:")[-1].strip()

# ── 3. Parse think + answer ──────────────────────────────────────
think_match = re.search(r"<think>(.*?)</think>", response_text, re.DOTALL)
reasoning   = think_match.group(1).strip() if think_match else ""
answer      = re.sub(r"<think>.*?</think>", "", response_text, flags=re.DOTALL).strip()

# ── 4. Extract structured fields ─────────────────────────────────
severity_match = re.match(r"^(Critical|High|Normal|Low|Borderline)[:\s]*([^.]*\.?)", answer, re.IGNORECASE)
severity       = severity_match.group(1).strip() if severity_match else "Unknown"
summary        = severity_match.group(2).strip() if severity_match else answer[:120]

reco_sentences = [
    s.strip() for s in re.split(r"(?<=[.!?])\s+", answer)
    if any(k in s.lower() for k in ["recommend", "suggest", "consider", "refer", "monitor", "prescribe", "urgent"])
]

test_name  = test_user.split(".")[0].strip()
result_val = (re.search(r"Result:\s*([\d.]+\s*\S+)", test_user) or re.search(r"", "")).group(1) if re.search(r"Result:\s*([\d.]+\s*\S+)", test_user) else ""
ref_range  = (re.search(r"Ref Range:\s*(.+)", test_user) or re.search(r"", "")).group(1).strip() if re.search(r"Ref Range:\s*(.+)", test_user) else ""

# ── 5. Build and print JSON ──────────────────────────────────────
result = {
    "lab_test": {
        "name":            test_name,
        "result":          result_val,
        "reference_range": ref_range,
    },
    "analysis": {
        "severity": severity,
        "summary":  summary,
        "full_clinical_response": answer,
    },
    "chain_of_thought":  reasoning,
    "recommendations":   reco_sentences,
}

print(json.dumps(result, indent=2))


{
  "lab_test": {
    "name": "Glucose (Fasting)",
    "result": "155 mg/dL.",
    "reference_range": "70-99 mg/dL."
  },
  "analysis": {
    "severity": "Critical",
    "summary": "High.",
    "full_clinical_response": "Critical: High. Extremely elevated glucose level indicative of severe insulin resistance or advanced diabetes mellitus. Immediate specialized treatment required. Recommend HbA1c testing and Oral Glucose Tolerance Test (OGTT) for definitive diagnosis. Consider checking Adiponectin and Lipid Panel as part of metabolic workup. Urgent referral to Endocrinology recommended. Suggest monitoring frequency based on patient risk factors and clinical management plan. Prescribe appropriate oral antidiabetic medications or Insulin therapy based on specific clinical scenarios. Ensure follow-up monitoring per standardized care protocols. Liver enzymes may also be checked; consider GPT/GDT levels if indicated by symptoms. Recommend dietary modifications under nutritional counseling gu

## 💾 Step 8: Save the Model

**Option A** — Save LoRA adapters only (small, ~150MB, requires base model for inference)

**Option B** — Save as merged 16-bit model (large, ~16GB, self-contained)

**Option C** — Push directly to Hugging Face Hub

In [17]:
NEW_MODEL_NAME = "Med-Lab-finetuned-DeepSeek-R1"
SAVE_PATH      = f"/content/{NEW_MODEL_NAME}"

# --- Option A: Save LoRA adapters only (recommended for quick saving) ---
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ LoRA adapters saved to: {SAVE_PATH}")

# --- Option B: Save merged 16-bit model (uncomment if needed) ---
# model.save_pretrained_merged(SAVE_PATH + '-merged', tokenizer, save_method='merged_16bit')
# print(f"✅ Merged model saved to: {SAVE_PATH}-merged")

# --- Option C: Push to Hugging Face Hub (uncomment & set your username) ---
HF_USERNAME = "Nguuma"
HF_REPO     = f"{HF_USERNAME}/{NEW_MODEL_NAME}"
model.push_to_hub(HF_REPO, token=hf_token)
tokenizer.push_to_hub(HF_REPO, token=hf_token)
print(f"✅ Model pushed to Hub: https://huggingface.co/{HF_REPO}")

✅ LoRA adapters saved to: /content/Med-Lab-finetuned-DeepSeek-R1
Saved model to https://huggingface.co/Nguuma/Med-Lab-finetuned-DeepSeek-R1
✅ Model pushed to Hub: https://huggingface.co/Nguuma/Med-Lab-finetuned-DeepSeek-R1


## 📥 Step 9: Download Saved Model (Optional)
Download the LoRA adapters as a zip file to your local machine.

In [18]:
import shutil
from google.colab import files

zip_path = f"/content/{NEW_MODEL_NAME}.zip"
shutil.make_archive(f"/content/{NEW_MODEL_NAME}", 'zip', SAVE_PATH)
print(f"✅ Archive created: {zip_path}")

files.download(zip_path)
print("📥 Download started.")

✅ Archive created: /content/Med-Lab-finetuned-DeepSeek-R1.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Download started.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Task
Merge the fine-tuned LoRA adapters with the base model and save the resulting model in GGUF format with `q4_k_m` quantization, then create a zip archive of the GGUF model and download it.

## Merge and Save as GGUF

### Subtask:
Merge the LoRA adapters with the base model and save the model in GGUF format with `q4_k_m` quantization.


**Reasoning**:
Following the instructions, I will now add a code block to define the GGUF save path and then call the `save_pretrained_gguf` method with the specified quantization to merge and save the model.



In [19]:
GGUF_SAVE_PATH = f"/content/{NEW_MODEL_NAME}-GGUF"

# Merge LoRA adapters and save as GGUF with q4_k_m quantization
# Note: Using quantization_method instead of quant_method
model.save_pretrained_gguf(
    GGUF_SAVE_PATH,
    tokenizer,
    quantization_method = "q4_k_m",
)

print(f"✅ Model merged and saved as GGUF to: {GGUF_SAVE_PATH}")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:52<00:00, 13.02s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:54<00:00, 13.52s/it]


Unsloth: Merge process complete. Saved to `/content/Med-Lab-finetuned-DeepSeek-R1-GGUF`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/Med-Lab-finetuned-DeepSeek-R1-GGUF_gguf/deepseek-r1-distill-llama-8b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/Med-Lab-finetuned-DeepSeek-R1-GGUF_gguf/deepseek-r1-distill-llama-8b.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/deepseek-r1-distill-llama-8b'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/Med-Lab-finetuned-DeepSeek-R1-GGUF_gguf/deepseek-r1-distill-llama-8b.Q4_K_M.gguf -p "why is the sky blue?"
✅ Model merged and saved as GGUF to: /content/Med-Lab-finetuned-DeepSeek-R1-GGUF


In [20]:
import shutil
from google.colab import files

# Create a zip archive of the GGUF model directory
gguf_zip_path = f"{GGUF_SAVE_PATH}.zip"
print(f"📦 Compressing {GGUF_SAVE_PATH}...")
shutil.make_archive(GGUF_SAVE_PATH, 'zip', GGUF_SAVE_PATH)

print(f"✅ Archive created: {gguf_zip_path}")

# Download the zip archive
files.download(gguf_zip_path)
print("📥 Download started.")

📦 Compressing /content/Med-Lab-finetuned-DeepSeek-R1-GGUF...
✅ Archive created: /content/Med-Lab-finetuned-DeepSeek-R1-GGUF.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Download started.
